# STAC API + COG (Kanopia) — Python demo

Goal (for a non-technical audience):
- Use a **STAC API** to find the *right* satellite products.
- Use a **COG** link to read only the small chunk of raster data we need (fast preview).

This notebook uses the Kanopia STAC API endpoint:
`https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/`

In [9]:
STAC_API_URL = "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac"

# Example query: around Quebec, September 2024
start_date = "2024-09-01T00:00:00Z"
end_date = "2024-10-01T23:59:59Z"
datetime_range = f"{start_date}/{end_date}"
keyword = "sbl"
limit = 50

bbox = [-79.76259, 45.00495, -57.10592, 62.58502]

print("Querying STAC:", STAC_API_URL)
print("  datetime:", datetime_range)
print("  bbox:", bbox)
print("  keyword:", keyword)
print("  limit:", limit)


Querying STAC: https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac
  datetime: 2024-09-01T00:00:00Z/2024-10-01T23:59:59Z
  bbox: [-79.76259, 45.00495, -57.10592, 62.58502]
  keyword: sbl
  limit: 50


In [11]:
# Optional: install Python dependencies for this notebook
from pathlib import Path
import sys
import subprocess

req_path = Path("/home/vincelf/vscode-workspaces/lefolab-utils/stac-api/notebooks/requirements.txt")
if not req_path.exists():
    raise FileNotFoundError(
        f"Could not find: {req_path}\n"
        "Run this notebook from the repo root, or edit `req_path` to point to requirements.txt."
    )

print("Installing:", req_path)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_path)])

print("Done. If imports fail afterwards, restart the notebook kernel.")


Installing: /home/vincelf/vscode-workspaces/lefolab-utils/stac-api/notebooks/requirements.txt
Defaulting to user installation because normal site-packages is not writeable
  Using cached rasterio-1.4.4-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.1.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached rasterio-1.4.4-cp311-cp311-manylinux_2_28_x86_64.whl (35.9 MB)
   ━━━━━━━


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Done. If imports fail afterwards, restart the notebook kernel.


In [12]:
import importlib.util
import sys

# Workshop mode: DO NOT auto-install.
# Participants will learn what they need to install.
AUTO_INSTALL = False

required = [
    ("requests",   "requests"),
    ("rasterio",   "rasterio"),
    ("matplotlib", "matplotlib"),
    ("numpy",      "numpy"),
]

missing = [
    pkg for (pkg, import_name) in required
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Missing Python packages:", missing)
    print("Install them in your shell, then restart the notebook/kernel.")
    print("  python3 -m pip install -r requirements.txt")
    print("Or (equivalent):")
    print("  python3 -m pip install " + " ".join(missing))
    raise SystemExit("Missing dependencies. Install and re-run.")
else:
    print("All required packages are already installed.")

import requests
import rasterio
from rasterio.enums import Resampling
import numpy as np
import matplotlib.pyplot as plt


All required packages are already installed.


In [13]:
def stac_search(stac_api_url, datetime_range, bbox, limit):
    """Query STAC /search endpoint."""
    payload = {
        "datetime": datetime_range,
        "bbox": bbox,
        "limit": limit,
    }
    r = requests.post(
        f"{stac_api_url}/search",
        json=payload,
        # Some environments have proxies that break POST; keep this explicit.
        proxies={"http": None, "https": None},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

def is_cog_asset(asset_key, asset_info):
    key = (asset_key or "").lower()
    mime = (asset_info.get("type") or "").lower()
    # COG-ish cues
    is_cogish = ("cloud-optimized" in mime) or ("cog" in key)
    return is_cogish and ("lowres" not in key) and ("overview" not in key)

data = stac_search(STAC_API_URL, datetime_range, bbox, limit)
features = data.get("features", [])

print("STAC response keys:", sorted(list(data.keys())))
print("features found:", len(features))

# Optional keyword filter on collection name
matched_features = []
for f in features:
    coll = (f.get("collection") or "").lower()
    if (keyword is None) or (keyword == "") or (keyword in coll):
        matched_features.append(f)

features = matched_features
print("Matched features after keyword filter:", len(features))


STAC response keys: ['features', 'links', 'numberReturned', 'type']
features found: 38
Matched features after keyword filter: 33


In [14]:
# Extract product/item assets that look like COGs
candidates = []

for f in features:
    assets = f.get("assets") or {}
    item_id = f.get("id") or f.get("properties", {}).get("id")
    collection = f.get("collection")

    for asset_key, asset_info in assets.items():
        if not asset_info:
            continue
        if is_cog_asset(asset_key, asset_info):
            href = asset_info.get("href")
            if href:
                candidates.append(
                    {
                        "href": href,
                        "asset_key": asset_key,
                        "collection": collection,
                        "item_id": item_id,
                    }
                )

print("Candidate COG assets:", len(candidates))

# Show a small preview list
for i, c in enumerate(candidates[:8]):
    print(f"[{i}] {c['asset_key']} | item_id={c['item_id']} | {c['href']}")

if not candidates:
    raise SystemExit("No COG-like assets found. Try changing keyword/limit or bbox.")


Candidate COG assets: 66
[0] dsm/optimized | item_id=20241001_sblz3_p1 | https://lab.kanopia.org/share/request-access/assets/2024/20241001_sblz3_p1/20241001_sblz3_p1_dsm_highdis.cog.tif
[1] rgb/optimized | item_id=20241001_sblz3_p1 | https://lab.kanopia.org/share/request-access/assets/2024/20241001_sblz3_p1/20241001_sblz3_p1_rgb.cog.tif
[2] dsm/optimized | item_id=20241001_sblz1z2_p1 | https://lab.kanopia.org/share/request-access/assets/2024/20241001_sblz1z2_p1/20241001_sblz1z2_p1_dsm_highdis.cog.tif
[3] rgb/optimized | item_id=20241001_sblz1z2_p1 | https://lab.kanopia.org/share/request-access/assets/2024/20241001_sblz1z2_p1/20241001_sblz1z2_p1_rgb.cog.tif
[4] dsm/optimized | item_id=20240930_sblz3_p1 | https://lab.kanopia.org/share/request-access/assets/2024/20240930_sblz3_p1/20240930_sblz3_p1_dsm_highdis.cog.tif
[5] rgb/optimized | item_id=20240930_sblz3_p1 | https://lab.kanopia.org/share/request-access/assets/2024/20240930_sblz3_p1/20240930_sblz3_p1_rgb.cog.tif
[6] dsm/optimized | i

## Pick an asset to preview

The STAC search returned a list of COG-like raster assets. The preview cell needs one of those asset links.

1. Look at the printed list above (`[0] ...`, `[1] ...`, ...).
2. Set `target_index` to the number you want to preview.

Tip: if no assets were found, try changing `keyword`, `limit`, or `bbox`.

In [15]:
# Choose which candidate to preview
target_index = 0  # change this to preview a different asset
target = candidates[target_index]

href = target["href"]
print("Previewing:")
print("  asset_key:", target["asset_key"])
print("  item_id:", target["item_id"])
print("  href:", href)

# Rasterio can read remote COGs efficiently via /vsicurl/
vsicurl_url = f"/vsicurl/{href}"

with rasterio.open(vsicurl_url) as src:
    print("\nCOG metadata:")
    print("  CRS:", src.crs)
    print("  Bounds:", src.bounds)
    print("  Size:", (src.width, src.height))
    print("  Bands:", src.count)
    print("  Driver:", src.driver)

    # Read a downsampled thumbnail from band 1
    band = 1
    out_h = 256
    out_w = int(src.width * (out_h / src.height))

    arr = src.read(
        band,
        out_shape=(out_h, out_w),
        resampling=Resampling.bilinear,
    )

arr = np.array(arr, dtype=float)
# Robust display range
finite = arr[np.isfinite(arr)]
vmin = np.percentile(finite, 2) if finite.size else 0
vmax = np.percentile(finite, 98) if finite.size else 1

plt.figure(figsize=(6, 6))
plt.imshow(arr, cmap="gray", vmin=vmin, vmax=vmax)
plt.title(f"COG preview: {target['asset_key']}")
plt.axis("off")
plt.show()


Previewing:
  asset_key: dsm/optimized
  item_id: 20241001_sblz3_p1
  href: https://lab.kanopia.org/share/request-access/assets/2024/20241001_sblz3_p1/20241001_sblz3_p1_dsm_highdis.cog.tif


RasterioIOError: HTTP response code: 404

### What just happened?

- **STAC API** search found candidate products in your time/area query.
- **COG** preview reads only a *small thumbnail* (fast), not the full raster file.

To re-run in your own workshop:
- Run cells top-to-bottom.
- Adjust `keyword` and `target_index` if needed.